<h1>Date and Time in Python</h1>

The most popular method for computers to count time is from an instant called <b>Unix epoch</b>, which is Jan 1, 1970 at 00:00:00 UTC (UTC is otherwise called GMT). Unix time elapses at the same rate as UTC. 

In old computers unix time since epoch is saved in 32-bit signed integers, which measn they will overflow on Jan 19, 2038.

In [5]:
import time

In [2]:
# seconds since epoch -- this is hard for humans to parse
time.time()

1688855159.3010502

Unix time is impossible to parse by humans, and is usually converted to UTC, which then can be converted to local time using any timezone offsets (so developers usually prefer saving the time in UTC which is usually enough to do any calculations later to retrieve when it was -- unless timezone rules changed after the saved date). 

Humans perceive time in years, months etc, which also add another level of complexity (locales). Again, the standard is that the dates should be written in the order of most-to-least-significant data, i.e. YYYY-MM-DD HH:MM:SS (so that there is no ambiguity, i.e. date first or month first).

Python's <b>datetime</b>, <b>calendar</b> and <b>time</b> libraries handles these complications. <b>calendar</b> which provides functions using Gregorian calendar, but returns instances of <b>datetime</b> too. 

<b>datetime</b> has <b>datetime.date</b> for dates, <b>datetime.time</b> for times (only 86400 seconds without a leap second, inlcudes <b>tzinfo</b>, goes up to microsecond) and <b>datetime.datetime</b> for date and time (has attributes of both classes).

In [1]:
import datetime

In [9]:
datetime.datetime(
    year=2022, month=1, day=31, hour=13, minute=14, second=31, microsecond=123456
)

datetime.datetime(2022, 1, 31, 13, 14, 31, 123456)

In [10]:
# Current local date
datetime.date.today()

datetime.date(2023, 7, 8)

In [140]:
# current local time
dt = datetime.datetime.now()
dt

datetime.datetime(2023, 7, 9, 23, 21, 33, 316333)

In [142]:
# 0-6 for Mon-Sun
dt.weekday()

6

<h2>Using strings</h2>

In [14]:
datetime.date.fromisoformat("2021-01-31")

datetime.date(2021, 1, 31)

In [23]:
# %Y - year in 4 digit
# %m - month as a zero-padded decimal, e.g. 01
# %d - day as a zero-padded decimal
# %H - hour as a zero-padded decimal with 24 hour clock
# %M - minutes as a zero-padded decimal
# %S - second as zero-padded decimal
# %f - milliseconds as zero-padded 6 digit decimal
datetime.datetime.strptime("01-31-2020 14:45:37", "%m-%d-%Y %H:%M:%S")

datetime.datetime(2020, 1, 31, 14, 45, 37)

In [26]:
dt = datetime.datetime(
    year=2022, month=1, day=31, hour=13, minute=14, second=31, microsecond=23456
)
# Always use ISO format (highest bit to lowest, so that strings can be compared too)
dt.strftime(format="%Y-%m-%d %H:%M:%S.%f")

'2022-01-31 13:14:31.023456'

<h2>Timezone-aware Date and Time</h2>

In [48]:
from zoneinfo import ZoneInfo
import pytz

In [105]:
unaware = datetime.datetime(
    year=2022, month=1, day=31, hour=13, minute=14, second=31, microsecond=23456
)
aware = unaware.astimezone(pytz.UTC)

In [106]:
aware.tzinfo

<UTC>

In [107]:
# Can't compare with unaware datetime
if unaware > aware:
    print("True")

TypeError: can't compare offset-naive and offset-aware datetimes

In [53]:
# some different types of timezones
tz1 = pytz.utc
tz2 = pytz.timezone("US/Eastern")
tz3 = datetime.timezone.utc
tz4 = ZoneInfo("America/New_York")

In [54]:
type(tz1), type(tz2), type(tz3), type(tz4)

(pytz.UTC, pytz.tzfile.US/Eastern, datetime.timezone, zoneinfo.ZoneInfo)

In [57]:
# They are different types but they all work with datetime
(
    datetime.datetime.now(tz=tz1),
    datetime.datetime.now(tz=tz2),
    datetime.datetime.now(tz=tz3),
    datetime.datetime.now(tz=tz4),
)

(datetime.datetime(2023, 7, 9, 17, 30, 56, 711168, tzinfo=<UTC>),
 datetime.datetime(2023, 7, 9, 13, 30, 56, 711181, tzinfo=<DstTzInfo 'US/Eastern' EDT-1 day, 20:00:00 DST>),
 datetime.datetime(2023, 7, 9, 17, 30, 56, 711196, tzinfo=datetime.timezone.utc),
 datetime.datetime(2023, 7, 9, 13, 30, 56, 711198, tzinfo=zoneinfo.ZoneInfo(key='America/New_York')))

In [90]:
# I will use pytz.timezone with datetime(tz=...) format going forward
ny_time = datetime.datetime.now(tz=pytz.timezone("America/New_York"))
ny_time

datetime.datetime(2023, 7, 9, 22, 37, 39, 423422, tzinfo=<DstTzInfo 'America/New_York' EDT-1 day, 20:00:00 DST>)

In [60]:
evn_time = datetime.datetime.now(tz=pytz.timezone("Asia/Yerevan"))
evn_time

datetime.datetime(2023, 7, 9, 21, 36, 43, 780098, tzinfo=<DstTzInfo 'Asia/Yerevan' +04+4:00:00 STD>)

In [97]:
type(ny_time.tzinfo), ny_time.tzinfo, ny_time.tzinfo.zone

(pytz.tzfile.America/New_York,
 <DstTzInfo 'America/New_York' EDT-1 day, 20:00:00 DST>,
 'America/New_York')

In [98]:
# what timezones are supported?
"Asia/Yerevan" in pytz.all_timezones

True

In [99]:
# Get the current timezone
# wraps a timedelta that is, go back 1 day from UTC, then go 72K seconds (20 hours) forward
# What is a timezone? Timedelta with a name!
local_tz = datetime.datetime.now().astimezone().tzinfo
type(local_tz), local_tz

(datetime.timezone,
 datetime.timezone(datetime.timedelta(days=-1, seconds=72000), 'EDT'))

In [104]:
local_tz.tzname(None)

'EDT'

<h3>Use time module?</h3>

In [65]:
# local timezone as tuple of 2 strings: local non-DST and local DST names
time.tzname

('EST', 'EDT')

In [66]:
time.localtime()

time.struct_time(tm_year=2023, tm_mon=7, tm_mday=9, tm_hour=14, tm_min=2, tm_sec=27, tm_wday=6, tm_yday=190, tm_isdst=1)

<h2>time:datetime conversions</h2>

In [128]:
# No matter where you are in the world, this returns seconds since UTC epoch
t = time.time()
t

1688958731.321156

In [129]:
# to conversion -- shows datetime in local timezone
dt = datetime.datetime.fromtimestamp(t)
dt

datetime.datetime(2023, 7, 9, 23, 12, 11, 321156)

In [130]:
# from
dt.timestamp()

1688958731.321156

In [132]:
# compare in seconds
t == dt.timestamp()

True

In [133]:
# Current time in UTC
dt = datetime.datetime.fromtimestamp(t, tz=pytz.utc)
dt

datetime.datetime(2023, 7, 10, 3, 12, 11, 321156, tzinfo=<UTC>)

In [134]:
# convenience method - nanos since UTC epoch
time.time_ns()

1688958961567189000

In [137]:
# another convenience method - time in UTC
utc_now = time.gmtime()
utc_now

time.struct_time(tm_year=2023, tm_mon=7, tm_mday=10, tm_hour=3, tm_min=17, tm_sec=49, tm_wday=0, tm_yday=191, tm_isdst=0)

In [139]:
# 0-6 for Mon-Sun
utc_now.tm_wday, utc_now.tm_min

(0, 17)

<h2>Time deltas</h2>

In [143]:
datetime.timedelta(days=10, weeks=1)

datetime.timedelta(days=17)

In [145]:
datetime.timedelta(
    weeks=1, days=1, hours=1, minutes=10, seconds=10, milliseconds=100, microseconds=200
)

datetime.timedelta(days=8, seconds=4210, microseconds=100200)

In [147]:
# only fixed periods are supported -> Year and month are not fixed
datetime.timedelta(years=1, months=2)

TypeError: 'years' is an invalid keyword argument for __new__()

In [148]:
dt = datetime.datetime.now()

In [151]:
future_dt = dt + datetime.timedelta(days=7)
# 1-7 for Mon-Sun
future_dt.isoweekday(), dt.isoweekday()

(7, 7)

In [152]:
# No new info to show for timezones, the only diff is, timezone is shown
dt_in_pacific = datetime.datetime.now(tz=pytz.timezone("US/Pacific"))
# Yesterday same time in Pacific
dt_in_pacific - datetime.timedelta(days=1) 

datetime.datetime(2023, 7, 8, 20, 31, 47, 527667, tzinfo=<DstTzInfo 'US/Pacific' PDT-1 day, 17:00:00 DST>)

<h2>Calendar</h2>

Use calendar to calculate various tasks based on date, month and year.

In [153]:
import calendar

In [157]:
c = calendar.TextCalendar(calendar.SUNDAY)

In [158]:
# Note, it starts from Sunday, but 1 is Wednesday
# Month calendar formatted string
c.formatmonth(2025, 1)

'    January 2025\nSu Mo Tu We Th Fr Sa\n          1  2  3  4\n 5  6  7  8  9 10 11\n12 13 14 15 16 17 18\n19 20 21 22 23 24 25\n26 27 28 29 30 31\n'

In [176]:
for i in c.itermonthdays(2025, 1):
    print(i, end=",")

0,0,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,0,

In [159]:
hc = calendar.HTMLCalendar(calendar.SUNDAY)]
hc.formatmonth(2025, 1)

In [177]:
for month in range(1, 13):
    cal = calendar.monthcalendar(2025, month)
    week1 = cal[0]
    week2 = cal[1]
    if week1[calendar.MONDAY] != 0:
        # First week has Monday in it
        auditday = week1[calendar.MONDAY]
    else:
        # First week has no Monday, audit on the 2nd week on Monday
        auditday = week2[calendar.MONDAY]
    print(f"{calendar.month_name[month]}: {auditday}th")

January: 6th
February: 3th
March: 3th
April: 7th
May: 5th
June: 2th
July: 7th
August: 4th
September: 1th
October: 6th
November: 3th
December: 1th


In [165]:
for name in calendar.month_name:
    print(name)


January
February
March
April
May
June
July
August
September
October
November
December


In [166]:
for name in calendar.day_name:
    print(name)

Monday
Tuesday
Wednesday
Thursday
Friday
Saturday
Sunday


In [168]:
calendar.MONDAY, calendar.SUNDAY

(0, 6)

In [178]:
calendar.January, calendar.February

(1, 2)

In [179]:
calendar.isleap(1992)

True